#TITLE:

Advanced Dimensionality Reduction & Data Augmentation for Image-Based ML
With Fashion MNIST — PCA, t-SNE, UMAP, Albumentations, and Classifier

# 🧾 HOW TO USE THIS IN COLAB

Open Google Colab

Start a new notebook

Enable GPU: Runtime → Change runtime type → Hardware Accelerator: GPU

Copy and paste each code block from this .docx file into a new Colab cell

Run the cells and modify as needed!

# 🛠️ INSTALL DEPENDENCIES


In [ ]:
!pip install -q umap-learn albumentations opentsne torchvision


In [ ]:
# 🔧 IMPORTS & SETUP
!pip install opentsne
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import torch
import torchvision
from torchvision import transforms
from tensorflow.keras.datasets import fashion_mnist
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap.umap_ as umap
from openTSNE import TSNE as OpenTSNE

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings("ignore")
sns.set(style='whitegrid')

In [ ]:
# 📥 LOAD AND PREPROCESS DATA
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
X = X_train.reshape((X_train.shape[0], -1))
X = StandardScaler().fit_transform(X)
y = y_train

label_dict = {
    0: "T-shirt", 1: "Trouser", 2: "Pullover", 3: "Dress", 4: "Coat",
    5: "Sandal", 6: "Shirt", 7: "Sneaker", 8: "Bag", 9: "Ankle boot"
}


In [ ]:
# 📊 PCA: Principal Component Analysis
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
print("Explained Variance Ratio:", pca.explained_variance_ratio_)


In [ ]:
# 📈 Visualize
def plot_2D(X_emb, y, title):
    df = pd.DataFrame({'x': X_emb[:, 0], 'y': X_emb[:, 1], 'label': y})
    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=df, x='x', y='y', hue='label', palette='tab10', alpha=0.7)
    plt.title(title)
    plt.legend()
    plt.show()

plot_2D(X_pca, y, "PCA - 2D Visualization")


In [ ]:
# 🧪 Scree Plot
pca_full = PCA().fit(X)
plt.figure(figsize=(8, 5))
plt.plot(np.cumsum(pca_full.explained_variance_ratio_), marker='o')
plt.title("Scree Plot")
plt.xlabel("Components")
plt.ylabel("Cumulative Explained Variance")
plt.grid(True)
plt.show()


Initially, the exercise loads and processes the Fashion-MNIST dataset for analysis. The training and test data are loaded using fashion_mnist.load_data(), and each 28×28 grayscale image is reshaped into a 784-dimensional vector. The data are standardized using StandardScaler to normalize the feature values so that features contribute equally and improve the algorithms’ performance later. The labels for the data are stored in y_train, with a mapping of numeric values to fashion categories T-shirt, Sneaker, and Ankle boot via a dictionary.

Then, Principal Component Analysis (PCA) is performed on the data to reduce its dimensions to the two principal components that explain the maximum variance in the data. The explained variance ratio is useful to show how much information from the original data is maintained after reducing the dimension of the vectors. Plot2D generates scatter plots of the two-dimensional vectors, with each class represented by a different color, allowing us to visualize separation of classes. A scree plot is also generated by fitting PCA on all of the components and plotting the cumulative explained variance against the number of components, indicating how many dimensions must be considered to retain most of the variability in the dataset.

In [ ]:
# 🌌 t-SNE with openTSNE (GPU-Friendly)
X_sample = X[:5000]
y_sample = y[:5000]

tsne = OpenTSNE(n_components=2, perplexity=30, n_jobs=4)
X_tsne = tsne.fit(X_sample)

plot_2D(X_tsne, y_sample, "t-SNE - 2D Projection")


The t-SNE visualization procedure utilizes the openTSNE library, which provides an accelerated and GPU-ready implementation of t-SNE for high-dimensional data. In order to reduce computation complexity, we select a subset of 5,000 samples (X_sample) and their respective labels (y_sample) from the Fashion-MNIST dataset. The t-SNE model has been specified for two-dimensional projection by setting n_components=2, and uses a perplexity of 30 to balance local and global structure within the data. Parallel computation is utilized for enhanced processing speed with n_jobs=4, an argument which will allow the use of four individual processors to calculate the 3D embeddings. The low-dimensional embeddings are calculated using the fit method, and assigned to a new variable called X_tsne. As part of the visualization step, we have created our own plotting function called plot2d. The plot2d function takes the two-dimensional t-SNE output we've created and creates a scatter plot of the embeddings, coloring each point by its class label. The result of this is a useful data visualization, providing insight into the spatial organization and cluster formation found in the data.

In [ ]:
# 🌐 UMAP
X_sample = X[:5000]
y_sample = y[:5000]

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1)
X_umap = reducer.fit_transform(X_sample)

plot_2D(X_umap, y_sample, "UMAP - 2D Projection")

The Uniform Manifold Approximation and Projection (UMAP) method is used to visualize the Fashion-MNIST using the reduced dimensionality technique. The UMAP reducer initializes with n_neighbors=15 to balance local and global structure preservation and min_dist=0.1 to maintain some distance in the dissimilar low-dimensional representation. The two-dimensional embedding is created using the fit_transform method and stored in X_umap. Then, plot_2D is used for visualization, which plots the scatter plot of the UMAP projection colored by class labels. This method shows the clustering of categories, which may also show patterns in terms of relationships between classes and intra-class similarities.


Data augmentation is also implemented as a way to diversify the dataset and improve the generalization of the model. This method comes with manually selecting five sample images and converting them into the PIL image format, taking advantage of some library features to run transformations like rotation, flipping, or scaling. The point of this strategy is to increase variability within the dataset, thereby improving the robustness of learning models.

In [ ]:
# 🧪 DATA AUGMENTATION
from PIL import Image
import albumentations as A
import cv2

# Sample images for augmentation
sample_images = X_train[:5]
pil_images = [Image.fromarray(img) for img in sample_images]

# TorchVision transforms
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.RandomRotation(30),
    transforms.RandomResizedCrop(28, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.5, contrast=0.5)
])

def show_augmented(images, transform_fn, title):
    fig, axs = plt.subplots(1, len(images), figsize=(15, 3))
    for i, img in enumerate(images):
        axs[i].imshow(transform_fn(img), cmap='gray')
        axs[i].axis('off')
    plt.suptitle(title)
    plt.show()

show_augmented(pil_images, transform, "Torchvision Basic Augmentations")


In [ ]:
# 🧪 Gaussian Noise
def add_gaussian_noise(img, std=0.2):
    noise = np.random.normal(0, std, (28, 28))
    noisy = np.clip(np.array(img)/255.0 + noise, 0, 1)
    return Image.fromarray((noisy * 255).astype(np.uint8))

show_augmented(pil_images, add_gaussian_noise, "Gaussian Noise Augmentation")


In [ ]:
# 🧪 Albumentations Pipeline
alb_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=40, p=0.8),
    A.RandomBrightnessContrast(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.2, rotate_limit=15, p=0.8),
    A.GaussianBlur(p=0.2)
])

def apply_albumentations(images, transform):
    fig, axs = plt.subplots(1, len(images), figsize=(15, 3))
    for i, img in enumerate(images):
        aug = transform(image=np.array(img))["image"]
        axs[i].imshow(aug, cmap='gray')
        axs[i].axis('off')
    plt.suptitle("Albumentations – Advanced Augmentations")
    plt.show()

apply_albumentations(pil_images, alb_transform)


In [ ]:
# ✅ CLASSIFIER ON REDUCED FEATURES
X_reduced = PCA(n_components=50).fit_transform(X[:10000])
y_reduced = y[:10000]

clf = LogisticRegression(max_iter=2000)
score = cross_val_score(clf, X_reduced, y_reduced, cv=5)
print("Accuracy (PCA-50):", score.mean())


Data augmentation is utilized to improve the variety of the Fashion-MNIST dataset, leading to increases in model robustness and generalization. This stage begins with five selected sample images and transforms them into a PIL image format for use with augmentation libraries. Several TorchVision transformations, including RandomHorizontalFlip, RandomRotation, RandomResizedCrop, and ColorJitter are performed to enable both geometric and color variations. The results of those augmentations are represented in a horizontal grid through the show_augmented function.

A user-defined function is then used for Gaussian noise to be added to the images while preserving the pixel value within valid limits. By adding this noise, the model is made less sensitive to small changes in the input. For more detailed transformations, an Albumentations pipline is then described, including HorizontalFlip, Rotate, RandomBrightnessContrast, ShiftScaleRotate, and GaussianBlur. The results of the augmented images are shown through the apply_albumentations reuseable function. Overall, by employing basic and advanced augmentation strategies, the dataset has a mixture of geometric, photometric, and noise variations to mimic the real world and improve the generalization patter of deep learning models.

🟦 PCA
Coding:
Train a classifier using PCA(100) components.

Compare performance with PCA(10).

Visualize PCA-transformed images in 3D.



In [ ]:
pca_100 = PCA(n_components=100)
X_pca100 = pca_100.fit_transform(X[:10000])
y_pca100 = y[:10000]

clf_100 = LogisticRegression(max_iter=2000)
score_100 = cross_val_score(clf_100, X_pca100, y_pca100, cv=5)
print(f"Accuracy (PCA-100): {score_100.mean():.4f}")


pca_10 = PCA(n_components=10)
X_pca10 = pca_10.fit_transform(X[:10000])

clf_10 = LogisticRegression(max_iter=2000)
score_10 = cross_val_score(clf_10, X_pca10, y_pca100, cv=5)
print(f"Accuracy (PCA-10): {score_10.mean():.4f}")

from mpl_toolkits.mplot3d import Axes3D

pca_3d = PCA(n_components=3)
X_pca3d = pca_3d.fit_transform(X[:2000])
y_pca3d = y[:2000]

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(X_pca3d[:, 0], X_pca3d[:, 1], X_pca3d[:, 2],
                     c=y_pca3d, cmap='tab10', alpha=0.7)

ax.set_title("3D PCA Visualization – Fashion MNIST")
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_zlabel("PC 3")
plt.legend(handles=scatter.legend_elements()[0], labels=list(label_dict.values()), loc="upper right")
plt.show()


Principal Component Analysis (PCA) is performed for dimensionality reduction of the Fashion-MNIST dataset and classification performance measurement using Logistic Regression. The first 10000 samples are drawn from the dataset, and reduced by PCA to 100 principal components. A Logistic Regression classifier is trained on them with 5-fold cross-validation, and mean accuracy is reported as a measure of predictive performance. In order to observe the effect of stronger dimensionality reduction, the same 10,000 samples are transformed again using only 10 principal components, and cross-validated accuracy is reported. This is to show how more extreme dimensionality reduction can bias the classification results.


Next, PCA is performed for three-dimensional plotting of the first 2000 samples. The first three principal components are projected into three-dimensional space, and the points are plotted using a scatter plot, coded by color according to the class labels. This serves to show the PCA data in a way that demonstrates how PCA captures variance in the data, and how much separation between classes occurred in a lower dimensional situation.

What does PCA optimize?
Principal Component Analysis (PCA) optimizes the variance in a dataset. PCA calculates new orthogonal directions that are called principal components that maximize the variance in the dataset. PCA projects the data onto these principal components, allowing compression of the dataset without losing too much information from the original dataset. PCA also defines the principal axes the data lies along and minimizes reconstruction error with for the reduced representation of the data. As a part of reducing the dimensionality of the data to create a lower dimensional representation of the data, PCA also maximizes the variance of the original dataset.

When can PCA hurt model performance?
PCA can hurt model performance when its assumptions are not valid for the data. Some important predictive feature may exist in the dataset, but because those features have low variance, PCA discards that important component for the lesser variance part of the data. Also, because the PCA is creating linear combinations of the original variables, the interpretability of the variables in the context for which they were used will be lost. This potential will become more critical to a field or discipline that requires transparency in the feature level. The linearity of PCA will also limit its ability to capture a complex non-linear relationship. Less important predictive features will also enter the model at the expense of better fitting components. The opposite is true when using too few components; therefore, the selection of components leads to underfitting and lost information. Under small or noise dataset, PCA identifies and amplifies noise, thereby distorting original patterns of the data, ultimately causing a decrease in model accuracy.

🌌 t-SNE
Coding:
Try different perplexities (5, 30, 50).

Apply t-SNE on PCA-reduced data.

Use t-SNE for only two classes.

In [ ]:
X_sample = X[:2000]
y_sample = y[:2000]

perplexities = [5, 30, 50]

for p in perplexities:
    tsne = OpenTSNE(n_components=2, perplexity=p, n_jobs=4, random_state=42)
    X_tsne = tsne.fit(X_sample)
    plot_2D(X_tsne, y_sample, f"t-SNE with Perplexity = {p}")


pca_50 = PCA(n_components=50).fit_transform(X_sample)
tsne_pca = OpenTSNE(n_components=2, perplexity=30, n_jobs=4, random_state=42)
X_tsne_pca = tsne_pca.fit(pca_50)

plot_2D(X_tsne_pca, y_sample, "t-SNE on PCA-Reduced Data (50 Components)")


mask = (y == 7) | (y == 9)
X_two = X[mask][:2000]
y_two = y[mask][:2000]

tsne_two = OpenTSNE(n_components=2, perplexity=30, n_jobs=4, random_state=42)
X_tsne_two = tsne_two.fit(X_two)

plot_2D(X_tsne_two, y_two, "t-SNE for Two Classes: Sneaker vs Ankle Boot")


The t-SNE algorithm is applied under multiple configurations to analyze its behavior and the resulting data structure. Initially, a subset of 2,000 samples from the Fashion-MNIST dataset is selected, and two-dimensional embeddings are generated using perplexity values of 5, 30, and 50. These variations in perplexity demonstrate how local and global cluster separations are affected by this parameter. Subsequently, Principal Component Analysis (PCA) is employed to reduce the same subset to 50 components before applying t-SNE. This step highlights how preliminary dimensionality reduction through PCA not only accelerates the t-SNE computation but also preserves the essential structure of the data. Finally, t-SNE is applied specifically to two fashion categories—Sneaker (class 7) and Ankle Boot (class 9)—to visualize and compare the separation between these closely related classes in the low-dimensional space.


Why t-SNE is Not Suitable for Classification
t-SNE (t-distributed Stochastic Neighbor Embedding) is mainly a tool for visualizing high-dimensional data, rather than a classification algorithm. It is used to project data into lower dimensions (usually 2D or 3D) and uncover clusters and relationships between samples, but it does not preserve class boundaries or global distances. t-SNE is a non-linear, non-parametric method, which means it does not learn a function that could generalize back to unlabeled data. Each run will independently re-embed the dataset and it is impossible to predict the label for a new sample. Additionally, t-SNE is very computationally expensive in use and is very sensitive to hyperparameters, which can produce unsubstantiated cluster sepation and unstable decision boundaries, which is problematic for classification.

The Role of Perplexity in t-SNE
Perplexity is a very important hyperparameter in t-SNE that mediates the tradeoff between local and global structure; it can be viewed as the effective number of neighbors to consider for measuring high-dimensional similarity. When the perplexity is low, it favors local relationships in which the clusters form tighter groups. On the other hand, high perplexity representatives broader global structure to preserve these relationships. If the perplexity is set too low or too high, it may provide poor visualizations. In these cases, it is advisable to stay focused on commonly accepted choices that have presented themselves useful purchases, which is a range, around 5-50; this is a starting point to tune and optimize based on the dataset.

🌐 UMAP
Coding:
Change n_neighbors and observe changes.

Use UMAP output for KMeans clustering.

Compare UMAP with PCA visually.

In [ ]:


from sklearn.cluster import KMeans


X_sample = X[:2000]
y_sample = y[:2000]

neighbors_list = [5, 15, 50]

for n in neighbors_list:
    reducer = umap.UMAP(n_neighbors=n, min_dist=0.1, random_state=42)
    X_umap = reducer.fit_transform(X_sample)
    plot_2D(X_umap, y_sample, f"UMAP with n_neighbors = {n}")


reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
X_umap = reducer.fit_transform(X_sample)

kmeans = KMeans(n_clusters=10, random_state=42)
cluster_labels = kmeans.fit_predict(X_umap)


plt.figure(figsize=(10, 7))
sns.scatterplot(x=X_umap[:, 0], y=X_umap[:, 1], hue=cluster_labels, palette='tab10', alpha=0.7)
plt.title("KMeans Clustering on UMAP Output")
plt.legend(title="Cluster", loc='best')
plt.show()


pca_2 = PCA(n_components=2).fit_transform(X_sample)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(ax=axes[0], x=pca_2[:, 0], y=pca_2[:, 1], hue=y_sample, palette='tab10', alpha=0.7)
axes[0].set_title("PCA - 2D Projection")

sns.scatterplot(ax=axes[1], x=X_umap[:, 0], y=X_umap[:, 1], hue=y_sample, palette='tab10', alpha=0.7)
axes[1].set_title("UMAP - 2D Projection")

plt.suptitle("Comparison: PCA vs UMAP")
plt.show()


A subset of 2,000 samples is selected and we run UMAP with various values of n_neighbors=5, 15, and 50, while fixing min_dist=0.1. We create two-dimensional embeddings and visualize the embeddings using the plot2D function to see the effects of different n_neighbors values (neighborhood sizes) on some combination of the local/global balance of structure seen in the projection. Next, we run UMAP using n_neighbors=15, and the resulting 2D embedding is used to run KMeans clustering with "n_clusters=10." We visualize the resulting clusters created by KMeans in a scatter plot and show how KMeans leveraged the structure identified by UMAP. Last, we visualize 2D PCA and 2D UMAP projections on the same subset for side-by-side comparison, indicating that UMAP better preserves both local and global relationships and shows better separation of classes in the lower-dimensional space than PCA.

How UMAP preserves global structure
UMAP preserves global structure by accounting for relationships among all points in the dataset, both nearby and distant. It constructs a high-dimensional graph in which a fuzzy topological representation encodes local neighborhoods as well as broader associations. During dimensionality reduction, UMAP optimizes the low-dimensional embedding to maintain the overall shape of the data and the relative distances between clusters. This approach allows it to simultaneously preserve fine details of local clusters and the general global structure of the dataset. In contrast, t-SNE emphasizes local relationships and often sacrifices global structure in favor of accurately representing local neighborhoods.

Why UMAP is faster than t-SNE
UMAP achieves higher computational efficiency through its graph-based approach and approximate nearest neighbor search. Its objective function is optimized using a simpler stochastic gradient descent method. On the other hand, t-SNE requires the computation of all pairwise similarities, which becomes increasingly time-consuming as dataset size grows. Consequently, UMAP can handle larger datasets while producing high-quality embeddings in significantly less time compared to t-SNE.

🧪 Augmentation
Coding:
Build a pipeline with 4 augmentations.

Apply augmentation only to class 9.

Show a grid of 10 random augmentations.

In [ ]:
# 🧪 Augmentation pipeline: 4 ops, apply only to class 9, show 10 random augments

import random
import matplotlib.pyplot as plt
import albumentations as A
import numpy as np

# pick class id for "Ankle boot"
target_class = 9

# 1) Define Albumentations pipeline (4 augmentations)
aug_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=30, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.GaussianBlur(blur_limit=(3, 7), p=0.4)
])

# 2) Collect indices of class 9 in training set
indices_class9 = np.where(y_train == target_class)[0]
print(f"Found {len(indices_class9)} images of class {target_class} (Ankle boot).")

# 3) Function to apply augmentation and return uint8 image
def augment_image(image_array):
    # image_array expected shape (28,28) with dtype uint8 or float in [0,255]
    img = image_array.astype(np.uint8)
    # Albumentations expects HWC; create single-channel HWC
    img_hwc = np.expand_dims(img, axis=2)
    aug = aug_pipeline(image=img_hwc)["image"]
    # convert back to 2D
    aug2d = aug[:, :, 0]
    return aug2d

# 4) Make 10 random augmented examples (randomly sample images of class 9 and augment)
random.seed(42)
np.random.seed(42)

num_examples = 10
sample_idx = np.random.choice(indices_class9, size=num_examples, replace=True)

augmented_images = [augment_image(X_train[i]) for i in sample_idx]

# 5) Show a 2x5 grid of augmented images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.ravel()
for ax, img in zip(axes, augmented_images):
    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    ax.axis('off')
plt.suptitle("10 Random Augmentations — Class 9 (Ankle boot)")
plt.tight_layout()
plt.show()


# OPTIONAL: If you want to create an augmented dataset (e.g., k augmentations per class-9 image)
k = 2  # number of augmentations per original class-9 image
aug_X = []
aug_y = []

for idx in indices_class9:
    orig = X_train[idx]
    for _ in range(k):
        aug_img = augment_image(orig)
        aug_X.append(aug_img)
        aug_y.append(target_class)

# Convert to arrays and (optionally) append to original dataset
aug_X = np.array(aug_X)  # shape (len(indices_class9)*k, 28, 28)
aug_y = np.array(aug_y)

print("Augmented set shape:", aug_X.shape, aug_y.shape)
# To append to training set (if desired):
# X_train_aug = np.concatenate([X_train, aug_X], axis=0)
# y_train_aug = np.concatenate([y_train, aug_y], axis=0)


A sample of 2,000 is chosen, and UMAP is used with different n_neighbors values of 5, 15, and 50 while keeping min_dist equal to 0.1. UMAP generates a 2D embedding which is visualized using the plot2D function in order to see how the various sizes of the neighborhood bias between local and global structure in the projections. Next, UMAP is run with n_neighbors equal to 15, and the resulting 2D embeddings are used for a KMeans algorithm converted to 10 clusters. The clusters are visualized in a scatter plot to show how KMeans is built off the structure generated by UMAP. Lastly, the 2D PCA and UMAP are done on the same subset to demonstrate how UMAP is able to keep the local and global relationships in space producing a more separated low-dimensional space in comparison to PCA.

Risks of excessive augmentation
Excessive augmentation takes place when the transformations applied create excessive distortion of the data, or create samples that are unrealistically different from the source data. Although augmentations generally increase dataset diversity, and improve model generalization, it is possible for improper, and/or excessive augmentations to confuse a model. For example, making large rotations of images, or performing extremely large flips, or performing radical color component changes could generate images that no longer fit with the true data distribution in the real world. Such issues will lead to overfitting to artificial artifacts and underfitting by hiding meaningful patterns (or some combination of both). Excessive augmentation can also waste resources, and training time, without improving model performance, and increases the risk of label noise if the augmented image no longer represents the original class.  

When to use GAN- augmentation
GANs (Generative Adversarial Networks) are particularly valuable for augmentation in data scenarios where the quantity of data is limited, or when high-quality synthetic samples are desired. GANs will create realistic images with class-specific features, thus they are particularly useful for including rare classes or imbalanced datasets. When compared to conventional augmentation (which tries to perform simple geometric, or photometric transformations), GAN augmentation will create entirely new samples, adding meaningful diversity to the dataset. In particular, high-dimensional, or complex visual datasets such as medical images, fashion items, or some other application where capturing realistic.